# Lab 05-06 — Pseudo-relevance feedback: expand the query with retrieved terms

**Track 05 · Query transformation** — every other lab in this track pays for query transformation with an LLM call. Pseudo-relevance feedback (PRF) is the LLM-free variant: it treats the retriever's own first pass as feedback. Stage 1 retrieves with the raw query, harvests the top terms from the top-k documents it found (minus stopwords and terms already in the query), appends them to the query, and retrieves again. The expansion terms pull stage 2 toward the vocabulary of the documents the first stage already judged relevant — no LLM, no training, no index rebuild.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, and faiss directly — no repo component library. Every block of the pipeline is built right here:

```
user question ──► BGE embed ──► FAISS top-3 ──► harvest top terms (re + Counter, no LLM)
                                                            │
                                                            ▼
expanded query (question + terms) ──► BGE embed ──► FAISS top-3
```

That is exactly how the shared component in `src/` works underneath: `src/tools/prf.py` is this same two-stage loop — a LangChain gap, hence the custom tool — and `src/retrieval/similarity.py` is the inner plain top-k retriever.

The lab compares, for the same questions:

* **RAW** — plain top-k: embed the question as the user typed it;
* **PRF** — two-stage: raw query -> harvest terms -> expanded query -> top-k.

Same three questions as labs 01/02/04/05 (1606/1610/1626) so you can compare the transformations directly. No LLM anywhere — embeddings are local BGE and every other step is pure Python (`re` + `Counter`), so the whole run is deterministic and fully offline.


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/passages.parquet` + `test.parquet`, already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `sentence-transformers`, and `faiss-cpu`. This is the one lab in the track with **no LLM anywhere** — no `.env`, no API key, no Groq. The bootstrap cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu pandas


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import re
import time
from collections import Counter
from pathlib import Path

import pandas as pd

# LangChain + sentence-transformers + faiss — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 100` takes a deterministic head of the 3200-passage corpus; `QUESTION_IDS = [1606, 1610, 1626]` reuses labs 01/02/04/05's questions so the transformations are directly comparable; `TOP_K = 3` is the stage-2 retrieval depth; `FEEDBACK_K = 3` is how many stage-1 documents the expansion terms are harvested from; `N_TERMS = 5` is how many terms the expanded query gains; `BGE_MODEL_NAME` pins the local embedder. `PREVIEW` truncates the passage previews the demo prints next to each hit. Together `FEEDBACK_K` and `N_TERMS` control how far stage 2 drifts from the raw query.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1626]  # same questions as labs 01/02/04/05, for comparison
TOP_K = 3  # stage-2 retrieval depth
FEEDBACK_K = 3  # how many stage-1 documents the expansion terms are harvested from
N_TERMS = 5  # how many terms the expanded query gains
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
PREVIEW = 62  # max characters of passage text shown next to each hit


## 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files

`load_passages` reads the first `n` passages (text + ids) from `passages.parquet`; `load_questions` pulls specific rows by id from `test.parquet`; `preview` flattens a passage onto one line for printing. Identical helpers to the other labs of this track keep the experiments directly comparable — the only thing that changes is the retriever wrapper.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 3. Experiment — raw vs PRF retrieval for the same questions

The whole pipeline is built inline. The embedder is `HuggingFaceEmbeddings` with the local BGE model (`normalize_embeddings=True`, which BGE requires for cosine); we embed the 100 passages once, then hand the FAISS store its vectors through a tiny precomputed passthrough — so the embed step and the index step stay separately timed, exactly like the lab. The inner retriever is a plain `similarity_search_by_vector` at `TOP_K = 3` (the inline shape of `src/retrieval/similarity.py`), and the PRF block is the same two-stage loop the shared `PRFRetriever` wraps: stage 1 retrieves with the raw query and keeps the top `FEEDBACK_K` documents; `_feedback_terms` harvests the top `N_TERMS` tokens from those documents (excluding stopwords and terms already in the query); stage 2 retrieves with the expanded query and returns the top `TOP_K`.

No LLM anywhere — the expansion is pure Python (`re` + `Counter`), so the whole run is deterministic and fully offline.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — raw vs PRF retrieval for the same questions
# --------------------------------------------------------------------------
PRF_STOPWORDS = frozenset(
    """a an and are as at be but by for from has have he her his how i if in
    is it its of on or she that the their them then there these they this to
    was were what when where which who why with you your""".split()
)


class _PrecomputedEmbeddings(Embeddings):
    """Hand the store precomputed vectors (looked up BY TEXT, not by order).

    FAISS calls embed_documents once with the full list; the lookup keeps the
    embed step and the index step separately timed, and would stay correct if
    a store batched the call.
    """

    def __init__(self, texts: list[str], embeddings: list[list[float]]):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        raise NotImplementedError("precomputed embeddings cannot embed queries")


class _SimilarityRetriever:
    """Inline stand-in for src/retrieval/similarity.SimilarityRetriever."""

    def __init__(self, store, embedder, top_k: int = TOP_K):
        self.store = store
        self.embedder = embedder
        self.top_k = top_k

    def retrieve(self, question: str) -> list[Document]:
        """Embed the question and return the top-k most similar documents."""
        query_embedding = self.embedder.embed_query(question)
        return self.store.similarity_search_by_vector(query_embedding, k=self.top_k)


class _PRFRetriever:
    """Inline stand-in for src/tools/prf.PRFRetriever."""

    def __init__(self, retriever, top_k: int = TOP_K, feedback_k: int = FEEDBACK_K,
                 n_terms: int = N_TERMS, min_term_len: int = 3,
                 stopwords: set[str] | None = None):
        self.retriever = retriever
        self.top_k = top_k
        self.feedback_k = feedback_k
        self.n_terms = n_terms
        self.min_term_len = min_term_len
        self.stopwords = stopwords if stopwords is not None else PRF_STOPWORDS

    def _tokenize(self, text: str) -> list[str]:
        """Lowercase alphanumeric tokens of ``text``."""
        return re.findall(r"[a-z0-9]+", text.lower())

    def _feedback_terms(self, question: str, docs: list[Document],
                        n_terms: int) -> list[str]:
        """Top ``n_terms`` tokens by frequency across ``docs``.

        Query terms, stopwords, and tokens shorter than ``min_term_len`` are
        excluded — an expansion term must add information the query does not
        already carry.
        """
        query_tokens = set(self._tokenize(question))
        counts: Counter[str] = Counter()
        for doc in docs:
            for tok in self._tokenize(doc.page_content):
                if (
                    tok in query_tokens
                    or tok in self.stopwords
                    or len(tok) < self.min_term_len
                ):
                    continue
                counts[tok] += 1
        return [tok for tok, _ in counts.most_common(n_terms)]

    def retrieve(self, question: str) -> list[Document]:
        """Stage-1 feedback, expand the query, stage-2 retrieval."""
        feedback = self.retriever.retrieve(question)[: self.feedback_k]
        terms = self._feedback_terms(question, feedback, self.n_terms)
        expanded = " ".join([question, *terms]) if terms else question
        return self.retriever.retrieve(expanded)[: self.top_k]


def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Embed locally (BGE) and index in-memory ---------------------------
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0

    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    t0 = time.perf_counter()
    store = FAISS.from_documents(chunks, embedding=_PrecomputedEmbeddings(passage_texts, passage_vecs))
    index_s = time.perf_counter() - t0

    # --- The two retrievers over the SAME store -----------------------------
    raw_retriever = _SimilarityRetriever(store, embedder, top_k=TOP_K)
    prf_retriever = _PRFRetriever(
        raw_retriever, top_k=TOP_K, feedback_k=FEEDBACK_K, n_terms=N_TERMS
    )

    # --- Per question: raw retrieval + the PRF expansion + stage-2 retrieval -
    results = []
    for qid, qtext in questions:
        raw_docs = raw_retriever.retrieve(qtext)

        # Replay PRF's two stages so the demo can show what changed.
        feedback = raw_retriever.retrieve(qtext)[:FEEDBACK_K]
        terms = prf_retriever._feedback_terms(qtext, feedback, N_TERMS)
        expanded = " ".join([qtext, *terms]) if terms else qtext

        t0 = time.perf_counter()
        prf_docs = prf_retriever.retrieve(qtext)
        prf_s = time.perf_counter() - t0

        results.append(
            {
                "qid": qid,
                "question": qtext,
                "terms": terms,
                "expanded": expanded,
                "prf_s": prf_s,
                "raw_docs": raw_docs,
                "prf_docs": prf_docs,
            }
        )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "indexed": len(passage_texts),
        "embed_s": embed_s,
        "index_s": index_s,
        "results": results,
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from three angles: the corpus subset with embedding/index timings; per question, the harvested feedback terms, the expanded query the stage-2 retriever actually saw, and the top-1 passage of both paths; then a takeaway on why PRF works: the expansion terms move the second query toward the vocabulary of what the first pass already judged relevant — cheap, deterministic, and free of API calls, at the price of amplifying whatever the first pass found, good or bad.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 06 — Pseudo-relevance feedback: expand the query with retrieved terms")
    print(f"{BGE_MODEL_NAME} (local) -> FAISS top-{TOP_K} -> PRF (no LLM)")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    embedded in {exp['embed_s']:.2f}s (dim 768), indexed in {exp['index_s']:.3f}s")

    print(f"\n[2] Raw vs PRF (per question):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["question"]}"')
        print(f"      feedback terms ({N_TERMS} max): {r['terms']}")
        print(f"      expanded query: {r['expanded']!r}")
        print(f"      raw  top-1: {preview(r['raw_docs'][0].page_content)}")
        print(f"      prf  top-1: {preview(r['prf_docs'][0].page_content)}")

    print("\n[3] Takeaway")
    print("    PRF is query expansion without an LLM: stage 1 retrieves with")
    print("    the raw query, stage 2 with the query plus the top terms of")
    print("    the stage-1 documents. The expansion terms move the second")
    print("    query toward the vocabulary of what the first pass already")
    print("    judged relevant — cheap, deterministic, and free of API calls.")
    print("    Trade-off vs the LLM-based labs: the terms come from retrieved")
    print("    text, so PRF amplifies whatever the first pass found, good or")
    print("    bad — garbage-in-garbage-out is stronger here.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: exactly `N_PASSAGES` passages indexed; every question returning `TOP_K` hits on both paths; every question harvesting >= 1 feedback term, with no expansion term appearing in the raw question (expansion must add information, not repeat it); and the content checks — the PRF top-3 must still carry the answer's keyword (montevideo / spanish / 1930). No LLM is involved, so this gate is fully deterministic — every run prints the same 12 PASS. This is the same gate the CI-style `--verify` run applies.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Structural properties (fully deterministic — no LLM anywhere).
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))
    checks.append(("each question returns TOP_K raw hits",
                   all(len(r["raw_docs"]) == TOP_K for r in exp["results"])))
    checks.append(("each question returns TOP_K PRF hits",
                   all(len(r["prf_docs"]) == TOP_K for r in exp["results"])))

    # The expansion must actually add terms, and they must be new information
    # (not stopwords, not the question's own tokens).
    for r in exp["results"]:
        tag = f"Q{r['qid']}"
        checks.append((f"{tag} harvested >= 1 feedback term", len(r["terms"]) >= 1))
        checks.append((f"{tag} expansion terms are not in the raw question",
                       not any(t in r["question"].lower() for t in r["terms"])))

    # Content checks: the expanded query must still surface the answer's
    # keyword. Q1606 -> Montevideo; Q1610 -> the Spanish; Q1626 -> 1930.
    for r in exp["results"]:
        tag = f"Q{r['qid']}"
        joined = " ".join(d.page_content for d in r["prf_docs"]).lower()
        if r["qid"] == 1606:
            kw = "montevideo"
        elif r["qid"] == 1610:
            kw = "spanish"
        else:  # 1626
            kw = "1930"
        checks.append((f"{tag} PRF top-{TOP_K} retains '{kw}'", kw in joined))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A minute of embedding + a handful of FAISS queries on the 100-passage subset — no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The three questions with the harvested feedback terms, the expanded query the stage-2 retriever actually saw, and the top-1 passage of the raw vs the PRF path. The expanded query is the whole lab — the raw question plus terms the first stage found relevant.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. No LLM is involved, so the gate is fully deterministic. If any line shows FAIL, check the parquet files are intact.


In [ ]:
verify_gate(exp)
